FAKE_NEWS_DETECTOR/
├── api/
│   ├── __init__.py
│   └── fast.py
│
├── data/
│   └── preprocessing.py
│
├── model/
│   ├── model.pkl
│   └── model.py
│
├── notebook/
│   ├── backup_py.ipynb
│   ├── charles.ipynb
│   └── test.ipynb
│
├── raw_data/
│   └── WELFake_Dataset.csv
│
├── streamlit/
│   ├── admin.py
│   ├── app.py
│   ├── chuck_norris.jpg
│   └── donald_trump.png
│
├── tests/
│
├── .gitignore
├── .python-version
├── Makefile
├── README.md
├── requirements.txt
└── utils.py

├── api/
│   ├── __init__.py
│   └── fast.py

__init__.py == fichier d'initialisation vide

fast.py (version tf-idf)

import pickle
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from data.preprocessing import clean

label_mapping = {0: "REAL", 1: "FAKE"}

class PredictRequest(BaseModel):
    text_to_analyze: str

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

with open("model/model.pkl", "rb") as f:
    app.state.model = pickle.load(f)

@app.get("/")
def root():
    return {"greeting": "Hello"}

@app.post("/predict")
def predict(request: PredictRequest):
    model = app.state.model
    cleaned = clean(request.text_to_analyze)

    predict_label = label_mapping[int(model.predict([cleaned])[0])]
    proba = model.predict_proba([cleaned])[0]
    predict_score = round(float(max(proba)), 4)

    return {"Verdict": str(predict_label), "Indice de confiance": predict_score}


#ajouter les feedbacks quand on les incluera


├── data/
│   └── preprocessing.py

preprocessing.py

import re
import string
from utils import data

def clean(text):

    text = text.strip().lower()
    for p in string.punctuation:
        text = text.replace(p, '')
    text = ' '.join(text.split(' '))
    text = re.sub('<[^<]+?>', '', text)
    text = text.replace('\n','')

    return text

def data_clean():

    df = data.copy()

    df = df.fillna("")
    df["article"] = df["title"] + " " + df["text"]

    df["article"] = df["article"].apply(clean)

    return df["article"]


├── model/
│   ├── model.pkl
│   └── model.py

model.pkl (modèle tf-idf entrainé)

model.py (version tf-idf)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from data.preprocessing import data_clean
from utils import data
import pickle

def init_model():

    model = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", LogisticRegression())
    ])

    return model

def train(model, X, y):

    model.fit(X,y)

    return model

def pred(model, text):

    return model.predict([text])[0]

def train_final():
    y = data['label']
    X = data_clean()

    model = init_model()
    model = train(model, X, y)

    print("modèle entraîné")

    return model


if __name__ == "__main__":
    print("START TRAIN")
    model = train_final()

    with open("model/model.pkl", "wb") as f:
        pickle.dump(model, f)


├── streamlit/
│   ├── admin.py
│   ├── app.py

admin.py

vide pour l'instant, accueillera le code de l'interface admin pour gestion des feedbacks

app.py (version tf-idf)

import streamlit as st
import requests
import os
from newspaper import Article


# params
if "url_count" not in st.session_state:
    st.session_state["url_count"] = 0

if "clear_count" not in st.session_state:
    st.session_state["clear_count"] = 0

img_chuck  = "streamlit/chuck_norris.jpg"
img_donald = "streamlit/donald_trump.png"
predict_label = ""
predict_score = 0.0
url_input = ""
text_input = ""
API_URL = os.getenv("FASTAPI_URL", "http://localhost:8000")
API_READY = True

st.set_page_config(
    page_title="Fake News Detector",
    page_icon="📰",
    layout="centered"
)

# css pour real/fake
st.markdown("""
<style>
    .verdict-fake {
        background-color: #ff4b4b22;
        border: 1px solid #ff4b4b;
        border-radius: 8px;
        padding: 16px;
        text-align: center;
        font-size: 1.4rem;
        font-weight: bold;
        color: #ff4b4b;
    }
    .verdict-real {
        background-color: #21c35422;
        border: 1px solid #21c354;
        border-radius: 8px;
        padding: 16px;
        text-align: center;
        font-size: 1.4rem;
        font-weight: bold;
        color: #21c354;
    }
</style>
""", unsafe_allow_html=True)

# header
st.title("📰 Fake News Detector 📰")
st.caption("Détection et catégorisation de Fake News")
st.divider()

# saisie
tab_url, tab_text = st.tabs(["Analyser depuis une URL", "Analyser depuis du texte brut"])

with tab_url:
    col_url, col_open = st.columns([5, 1])
    with col_url:
        url_input = st.text_input(
            "URL de l'article",
            placeholder="https://www.reuters.com/...",
            key=f"url_input_{st.session_state['url_count']}",
        )
    with col_open:
        st.markdown("<br>", unsafe_allow_html=True)
        if url_input:
            st.link_button("🔗 Ouvrir", url_input, use_container_width=True)
        else:
            st.button("🔗 Ouvrir", disabled=True, use_container_width=True)

with tab_text:
    text_input = st.text_area(
        "Collez ici le contenu de l'article",
        height=200,
        placeholder="Entrez le texte de l'article à analyser...",
        key=f"input_text_{st.session_state['clear_count']}",
    )

# bouton
st.divider()
col_btn, col_clear = st.columns([5, 1])
with col_btn:
    analyze = st.button("🔍 Lancer l'analyse", type="primary", use_container_width=True)
with col_clear:
    if st.button("🗑️ Effacer", use_container_width=True):
        st.session_state["clear_count"] += 1
        st.session_state["url_count"] += 1
        st.session_state["show_cleared"] = True
        st.rerun()

if st.session_state.get("show_cleared"):
    st.success("🗑️ Champs effacés")
    st.session_state["show_cleared"] = False

# analyse
if analyze:

    # cas 1 : url
    if url_input:
        with st.spinner("Extraction en cours..."):
            try:
                art = Article(url_input)
                art.download()
                art.parse()
                text_to_analyze = art.text
                st.success(f"Le texte analysé fait {len(text_to_analyze)} caractères")

            except Exception as e:
                st.error(f"❌ Impossible d'extraire le texte : {e}")
                st.stop()

    # cas 2 : texte brut
    elif text_input:
        text_to_analyze = text_input
        st.success(f"Le texte analysé fait {len(text_to_analyze)} caractères")

    # cas 3 : rien
    else:
        st.error("❌ Aucun texte à analyser. Collez du texte ou entrez une URL.")
        st.stop()

    # validation longueur
    if len(text_to_analyze.strip()) < 200:
        st.error("❌ Texte trop court. Veuillez saisir au moins 200 caractères.")
        st.stop()

    # prédiction
    with st.spinner("Analyse en cours..."):
        if API_READY:
            try:
                response = requests.post(
                    f"{API_URL}/predict",
                    json={"text_to_analyze": text_to_analyze},
                    timeout=10,
                )
                response.raise_for_status()
                result = response.json()
                predict_label  = result["Verdict"]
                predict_score  = result["Indice de confiance"]

            except requests.exceptions.ConnectionError:
                st.error(f"❌ API non joignable sur {API_URL}")
                st.stop()
            except Exception as e:
                st.error(f"❌ Erreur API : {e}")
                st.stop()

# résultat
    st.divider()
    st.subheader("Résultat")

    if predict_label == "FAKE":
        st.markdown(f'<div class="verdict-fake"><span style="font-size: 4rem;">🚨 FAKE NEWS</span><br><br>Indice de confiance : {predict_score:.1%}</div>', unsafe_allow_html=True)
    else:
        st.markdown(f'<div class="verdict-real"><span style="font-size: 4rem;">✅ ARTICLE FIABLE</span><br><br>Indice de confiance : {predict_score:.1%}</div>', unsafe_allow_html=True)


# feedback

#footer
st.divider()
st.caption("Martin Cornud - Alex Delrieu - Charles Jégo")
st.caption("Le Wagon - #2251")
